<a href="https://colab.research.google.com/github/interpretune/examples/blob/main/tutorials/publish/shared_analysis/shared_analysis_roundtrip.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

# Shared-Analysis Round-Trip: publish & replicate attribution-graph steering

This notebook demonstrates end-to-end shared analysis: one user generates attribution-graph
sign-aware feature-steering results; a second user replicates those results from the published
artifact with the expensive attribution-graph generation skipped entirely. Only the cheap steering forward pass re-runs from the stored graph.

1. Generate locally: the `it.intervention_from_concept` pipeline produces the reference
   results and an `AnalysisStore`.
2. Push:  the store publishes as a Hub *dataset* repo with a machine-written
   `it_artifact.json` envelope (parquet interchange; the local format stays Arrow).
3. Clean:  the local analysis state is deleted; only small reference snapshots remain.
4. Replicate:  `it.hub.pull_analysis_store` fetches the artifact, the interpretune formatter
   re-attaches from the envelope, and the named analysis backend hydrates steering-capable
   graphs; the steering results are reproduced and compared against the reference.
5. Optional coda:  a small provenance iteration re-uploads to the same repo: the artifact's
   `identity` survives, its content fingerprint tracks the change.


In [2]:
# Parameters - These will be injected by papermill during parameterized test runs
BACKEND = "nnsight"  # circuit-tracer backend: "nnsight" or "transformerlens"
CONCEPT_PROMPT = "Is orange a color or a fruit? Answer with one word: Color or Fruit. orange ->"
CONCEPT_TARGET_TOKENS = ["Fruit", "Color"]
FEATURE_SELECTION_TOP_N = 5
FEATURE_SELECTION_MIN_LAYER = 10
FEATURE_SELECTION_SCORE_SIGN = "any"
INTERVENTION_SCALE_FACTOR = 20.0
PROMPT_ANALYSIS_REVISION = "6dd0dd5260bdd56a2c8cd7fb8a2c377554288034"  # pinned: trusted hub code must not change
PROMPT_CONFIGS_REVISION = "3d8b9710061283dd14c86a46f0c0649b5e262db0"  # pinned: trusted hub code must not change
REGISTRY_KEY = "prompt_analysis.gemma2.circuit_tracer"  # hub component configuration key (model + backend)
MODEL_NAME = "gemma-2-2b"
TRANSCODER_SET = "gemma"
NEURONPEDIA_MODEL_ID = "gemma-2-2b"  # public dashboard substrate: feature indices link to neuronpedia.org
NEURONPEDIA_SOURCE_SET = "gemmascope-transcoder-16k"
CHAT_FORMAT_PROMPT = False

# -- The shared-analysis artifact -----------------------------------------------------------------
# Set this to YOUR OWN `<org>/<repo>` (any org you can write to). `speediedan/...` is only the
# maintainer's validation target -- nothing about the workflow is specific to it.
ARTIFACT_REPO_ID = "speediedan/ct-concept-steering-demo"
ARTIFACT_PRIVATE = False  # the maintainer's demo store is public so anyone can stream it; True keeps yours private
HF_TOKEN_ENV = "IT_HF_TOKEN"  # env var holding a WRITE token for ARTIFACT_REPO_ID's org
STREAM_COLUMNS = ["input_string", "logit_diff"]  # step 6: the columns a streamed read projects
RUN_REUPLOAD_CODA = True  # step 5: provenance-iteration re-upload (identity must survive)

In [3]:
# @title Imports { display-mode: "form" }
import os

import torch  # noqa: F401

import interpretune.analysis  # noqa: F401  # ensure op wrappers are registered
from interpretune.analysis import analysis_store_from_batches
from interpretune.analysis.backends import FeatureSelectionSpec
from interpretune.analysis.ops.base import AnalysisBatch
from interpretune.harness.notebook_ui import (  # noqa: F401
    display_steering_results,
    display_target_gap,
)

## 1. Generate locally:  the reference results

In [4]:
# @title 1: Session construction { display-mode: "form" }
from pathlib import Path

from dotenv import load_dotenv

import interpretune as it
from interpretune import ITSession, ITSessionConfig

for _env_candidate in (Path.cwd() / ".env", Path.home() / "repos" / "interpretune" / ".env"):
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        break

# fetch the component at its pinned revision (executes nothing), then opt in to running it, for this session only
it.hub.pull("speediedan/prompt-analysis", revision=PROMPT_ANALYSIS_REVISION)
os.environ["IT_TRUST_REMOTE_CODE"] = "1"
it.hub.pull("speediedan/prompt-analysis", REGISTRY_KEY, revision=PROMPT_ANALYSIS_REVISION)
base_itdm_cfg, base_it_cfg, dm_cls, m_cls = it.hub.load(
    "speediedan/prompt-analysis", REGISTRY_KEY, revision=PROMPT_ANALYSIS_REVISION
)
base_it_cfg.circuit_tracer_cfg.backend = BACKEND
if TRANSCODER_SET:
    base_it_cfg.circuit_tracer_cfg.transcoder_set = TRANSCODER_SET
if BACKEND == "nnsight":
    adapter_ctx = (it.Adapter.core, it.Adapter.nnsight, it.Adapter.circuit_tracer)
else:
    adapter_ctx = (it.Adapter.core, it.Adapter.transformer_lens, it.Adapter.circuit_tracer)
session_cfg = ITSessionConfig(
    adapter_ctx=adapter_ctx,
    datamodule_cfg=base_itdm_cfg,
    module_cfg=base_it_cfg,
    datamodule_cls=dm_cls,
    module_cls=m_cls,
)
it_session = ITSession(session_cfg)
it.it_init(**it_session)
module = it_session.module
tokenizer = module.replacement_model.tokenizer
print(f"session ready: {type(module).__name__} ({MODEL_NAME} + circuit-tracer {BACKEND} backend)")

src/interpretune/config/shared.py:281: Could not find an auto-composition for <class 'interpretune.config.module.ITConfig'> that supports all of the following kwargs: {'tl_cfg': ITLensFromPretrainedNoProcessingConfig(move_to_device=True, model_name='gemma-2-2b', fold_ln=False, center_writing_weights=False, center_unembed=False, refactor_factored_attn_matrices=False, checkpoint_index=None, checkpoint_value=None, hf_model=None, device='cuda', n_devices=1, tokenizer=None, fold_value_biases=False, default_prepend_bos=True, dtype='float32'), 'circuit_tracer_cfg': CircuitTracerConfig(backend='nnsight', model_name=None, transcoder_set='gemma', dtype=torch.bfloat16, max_n_logits=10, desired_logit_prob=0.95, batch_size=256, max_feature_nodes=8192, offload='cpu', freeze_attention=True, freeze_norms=True, lazy_encoder=None, lazy_decoder=True, lazy_delta_rank=None, tune_base=False, verbose=True, default_node_threshold=0.8, default_edge_threshold=0.98, save_graphs=True, graph_output_dir=None, analy

[INFO] interpretune.utils.logging: Loading ReplacementModel with backend: nnsight


[INFO] interpretune.utils.logging: NNsight ReplacementModel initialized for Circuit Tracer


src/interpretune/adapters/circuit_tracer/adapter.py:236: Lazily loaded transcoder weights (transcoders.0.W_enc, transcoders.0.W_dec, transcoders.1.W_enc and 49 more) are not parameters, so they are left out of this composition's tunable view; set `CircuitTracerConfig.lazy_delta_rank` to tune them through low-rank corrections, or load them eagerly (`lazy_encoder=False`, `lazy_decoder=False`).
[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'Gemma2Config' object has no attribute 'quantization_config'


[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'Gemma2Config' object has no attribute '_pre_quantization_dtype'


src/interpretune/session.py:322: CircuitTracerModule is not interpretune.protocol.BuiltTrainSteppable | interpretune.protocol.BuiltValidationSteppable | interpretune.protocol.BuiltTestSteppable | interpretune.protocol.BuiltPredictSteppable even after composing <class 'interpretune.adapters.circuit_tracer.adapter.CircuitTracerModule'> with <class 'interpretune.session.InterpretunableModule'>. This Interpretune session may not work as intended.
[INFO] interpretune.utils.logging: Preparing data: InterpretunableDataModule


[INFO] interpretune.utils.logging: Setting up datamodule: InterpretunableDataModule


[INFO] interpretune.utils.logging: Setting up model: InterpretunableModule


session ready: InterpretunableModule (gemma-2-2b + circuit-tracer nnsight backend)


In [5]:
# @title 2: Run the pipeline; capture the reference; build the store { display-mode: "form" }
from interpretune.config import AnalysisCfg, init_analysis_cfgs

module.analysis_cfg = AnalysisCfg(target_op=it.intervention_from_concept, ignore_manual=True, save_tokens=False)
init_analysis_cfgs(module, [module.analysis_cfg])

fruits = ["apple", "banana", "grape", "peach"]
colors = ["red", "blue", "green", "yellow"]
if CHAT_FORMAT_PROMPT:
    from interpretune.hub.promptconfigs import resolve_prompt_config_class

    it.hub.pull("speediedan/prompt-configs", revision=PROMPT_CONFIGS_REVISION)
    GemmaPromptConfig = resolve_prompt_config_class("speediedan/prompt-configs#GemmaPromptConfig")

    prompt = GemmaPromptConfig().apply_chat_template_fn(
        tokenizer, CONCEPT_PROMPT, tokenize=False, add_generation_prompt=True
    )
else:
    prompt = CONCEPT_PROMPT

ct_cfg = module.it_cfg.circuit_tracer_cfg
ct_cfg.intervention_sign_aware_scale = True
ct_cfg.intervention_max_influence_norm_scale = True
ct_cfg.intervention_value_source = "top_feature_activation_values"

selection_spec = FeatureSelectionSpec(
    layer_slice=slice(FEATURE_SELECTION_MIN_LAYER, None),
    score_source="signed_influence",
    score_sign=FEATURE_SELECTION_SCORE_SIGN,
    rank_by_abs=True,
)
pipeline_results = it.intervention_from_concept(
    module,
    AnalysisBatch(
        concept_group_a=fruits,
        concept_group_b=colors,
        concept_label="Concept: Fruit - Color",
        concept_direction_mode="paired_rejection",
        concept_basis="embed",
        prompts=[prompt],
    ),
    None,
    0,
    top_n=FEATURE_SELECTION_TOP_N,
    intervention_scale_factor=INTERVENTION_SCALE_FACTOR,
    feature_selection=selection_spec,
)
steering = display_steering_results(
    pipeline_results,
    tokenizer,
    CONCEPT_TARGET_TOKENS,
    neuronpedia_model=NEURONPEDIA_MODEL_ID,
    neuronpedia_set=NEURONPEDIA_SOURCE_SET,
)

# the store IS the shareable artifact: the pipeline results serialized by the op's own schema
store = analysis_store_from_batches(
    module, [pipeline_results], op=it.intervention_from_concept, analysis_backend="circuit_tracer"
)
# REFERENCE snapshots (plain python; everything else local is deleted in step 3)
reference_row = {k: v for k, v in store.dataset.with_format(None)[0].items()}
reference_columns = sorted(store.dataset.column_names)
print(f"reference captured: {len(reference_columns)} columns, 1 row")

Phase 0: Precomputing activations and vectors


Precomputation completed in 5.67s


Found 18348 active features


Phase 1: Running forward pass


Forward pass completed in 0.28s


Phase 2: Building input vectors


... (41 outputs left out)

Attribution completed in 1570.28s


#,Node,Sign,|Score|
1,"(25, 19, 16131)",−,1.94e-08
2,"(24, 19, 13277)",−,1.81e-08
3,"(24, 19, 5999)",+,9.92e-09
4,"(24, 19, 3865)",+,4.87e-09
5,"(25, 19, 13210)",+,4.74e-09


Token,Pre prob,Post prob,Pre logit,Post logit,Δ
Fruit,2.317%,1.861%,25.1250,29.3750,+4.2500
Color,15.106%,1.03e-05,27.0000,21.8750,-5.1250
Gap (Fruit − Color),,,-1.8750,+7.5000,+9.3750


reference captured: 44 columns, 1 row


## 2. Push the store to the Hub (parquet interchange + envelope + generated card)

In [6]:
# @title 3: Push { display-mode: "form" }
token = os.environ[HF_TOKEN_ENV]
pushed_revision = it.hub.push_analysis_store(
    store,
    ARTIFACT_REPO_ID,
    private=ARTIFACT_PRIVATE,
    token=token,
    provenance={"generated_by": "intervention_from_concept", "backend": BACKEND},
    # what the columns were computed on (model, tokenizer, component map, dictionaries): the store was assembled
    # from batches rather than written by an op on the module, so it is read from the module here
    subject=it.hub.subject_from_module(module),
)
print(f"pushed {ARTIFACT_REPO_ID} @ {pushed_revision}")

Setting num_proc from 1 back to 1 for the validation split to disable multiprocessing as it only contains one shard.


pushed speediedan/ct-concept-steering-demo @ 828fb22bfa48208eb16af8eb38590d098f1636d7


## 3. Clean the local analysis state

Everything the pipeline produced locally is deleted:  what follows must come from the Hub.

In [7]:
# @title 4: Clean { display-mode: "form" }
import gc
import shutil

local_save_dir = getattr(getattr(module.analysis_cfg, "output_store", None), "save_dir", None)
del store, pipeline_results
gc.collect()
if local_save_dir and Path(local_save_dir).exists():
    shutil.rmtree(local_save_dir)
    print(f"removed local analysis output: {local_save_dir}")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("local analysis state cleaned -- only the reference snapshots remain in memory")

removed local analysis output: /tmp/20261124_193633_default/analysis_datasets
local analysis state cleaned -- only the reference snapshots remain in memory


## 4. Replicate from the Hub artifact:  no pipeline re-run

`pull_analysis_store` fetches envelope-first (revision-pinned), re-attaches the interpretune
formatter from the envelope, and resolves the named analysis backend (`circuit_tracer`) so
row access hydrates steering-capable `Graph` objects. We verify three things:

1. Transport fidelity:  every stored column round-trips the parquet interchange intact.
2. Hydration:  the pulled store yields a live attribution `Graph` via the backend seam.
3. Steering replication:  the graph computation is skipped; `it.intervention_from_features`
   re-runs only the (cheap) steering forward pass from the stored graph and reproduces the
   reference steering outcome. The compute the second user avoids is the expensive
   attribution analysis.

In [8]:
# @title 5: Pull + verify { display-mode: "form" }
import numpy as np

pulled = it.hub.pull_analysis_store(ARTIFACT_REPO_ID, token=token)
envelope = it.hub.describe_analysis_store(ARTIFACT_REPO_ID)
print(f"artifact identity: {envelope['identity']['store_id']} (created {envelope['identity']['created_utc']})")
print(f"provenance: {envelope['provenance']}")

# 1) transport fidelity: every column byte-faithful through parquet
pulled_row = {k: v for k, v in pulled.dataset.with_format(None)[0].items()}
assert sorted(pulled.dataset.column_names) == reference_columns
mismatched = []
for col in reference_columns:
    ref, got = reference_row[col], pulled_row[col]
    equal = np.array_equal(np.asarray(ref, dtype=object), np.asarray(got, dtype=object))
    if not equal:
        mismatched.append(col)
assert not mismatched, f"columns changed in transport: {mismatched}"
print(f"transport fidelity: {len(reference_columns)}/{len(reference_columns)} columns identical")

# 2) hydration via the named backend seam (no generation pipeline involved)
hydrated = pulled[0]
graph = hydrated["attribution_graph"]
print(f"hydrated graph: {type(graph).__name__} ({graph.active_features.shape[0]} active features)")

# 3) steering replication: ONLY the intervention forward pass re-runs, from hydrated state
# the prompt itself travels IN the artifact (the graph's input_string) -- the second user
# needs nothing beyond the hub repo to re-run the steering forward
replicated = it.intervention_from_features(
    module,
    AnalysisBatch(**{k: v for k, v in hydrated.items()}, prompts=[graph.input_string]),
    None,
    0,
    intervention_scale_factor=INTERVENTION_SCALE_FACTOR,
)
replicated_steering = display_steering_results(
    replicated,
    tokenizer,
    CONCEPT_TARGET_TOKENS,
    neuronpedia_model=NEURONPEDIA_MODEL_ID,
    neuronpedia_set=NEURONPEDIA_SOURCE_SET,
)
print("steering replicated: graph computation skipped; steering ran from the stored graph")

artifact identity: aadcd8d6-3892-475e-9f8f-1cb21cde2305 (created 2026-08-11T23:35:47+00:00)
provenance: {'backend': 'nnsight', 'content_fingerprint': 'a8a753ecceaa52f0f27452bf2f2fcfbf4b906f2adc1a5654e04043c97c4dc9bd', 'generated_by': 'intervention_from_concept', 'interpretune_version': '0.1.0.dev821+g0cd95d639'}


transport fidelity: 44/44 columns identical


hydrated graph: Graph (18348 active features)


#,Node,Sign,|Score|
1,"(25, 19, 16131)",−,1.94e-08
2,"(24, 19, 13277)",−,1.81e-08
3,"(24, 19, 5999)",+,9.92e-09
4,"(24, 19, 3865)",+,4.87e-09
5,"(25, 19, 13210)",+,4.74e-09


Token,Pre prob,Post prob,Pre logit,Post logit,Δ
Fruit,2.317%,1.885%,25.1250,29.3750,+4.2500
Color,15.106%,1.04e-05,27.0000,21.8750,-5.1250
Gap (Fruit − Color),,,-1.8750,+7.5000,+9.3750


steering replicated: graph computation skipped; steering ran from the stored graph


## 5. Stream instead of downloading

The pull above downloaded every Parquet file. A reader who wants a few columns, or wants to start before the rest
arrives, can stream the store instead: `streaming=True` fetches only the envelope up front, then reads the Parquet files
the envelope lists, at its commit, through range requests, and `columns=` projects the read so the other columns are
never fetched. The rows decode exactly as the full pull's do.

This store holds one attribution graph in a single row, so almost all of its bytes are the graph's adjacency matrix.
Streaming the two small columns below transfers a small fraction of the file. To show how small, the cell counts the
range requests `HfFileSystem` makes while it reads; the counting is only instrumentation and is removed afterwards.

In [9]:
# @title 6: Stream a projection { display-mode: "form" }
import torch
from huggingface_hub import HfApi
from huggingface_hub.hf_file_system import HfFileSystemFile

transferred = {"requests": 0, "bytes": 0}
_fetch_range = HfFileSystemFile._fetch_range


def _counting_fetch(self, start, end):  # one call per range request
    data = _fetch_range(self, start, end)
    transferred["requests"] += 1
    transferred["bytes"] += len(data)
    return data


HfFileSystemFile._fetch_range = _counting_fetch
try:
    streamed = it.hub.pull_analysis_store(ARTIFACT_REPO_ID, streaming=True, columns=STREAM_COLUMNS, token=token)
    streamed_row = next(iter(streamed))
finally:
    HfFileSystemFile._fetch_range = _fetch_range

for col in STREAM_COLUMNS:
    got, want = streamed_row[col], pulled_row[col]  # the full pull's stored values, before hydration
    got = got.tolist() if isinstance(got, torch.Tensor) else got
    assert got == want, f"streamed {col!r} differs from the full pull"
info = HfApi(token=token).dataset_info(ARTIFACT_REPO_ID, files_metadata=True)
published = sum(f.size for f in info.siblings if f.rfilename.endswith(".parquet"))
print(f"streamed {STREAM_COLUMNS}: values match the full pull")
print(
    f"transferred {transferred['bytes']:,} bytes in {transferred['requests']} range requests, "
    f"of {published:,} published ({transferred['bytes'] / published:.2%})"
)

streamed ['input_string', 'logit_diff']: values match the full pull
transferred 297,334 bytes in 5 range requests, of 31,910,643 published (0.93%)


## 6. Optional coda:  share an iteration

Re-uploading after a change keeps the artifact's identity (`store_id` never rewritten) while
its provenance:  including the content fingerprint,  refreshes.

In [10]:
# @title 7: Re-upload coda { display-mode: "form" }
if RUN_REUPLOAD_CODA:
    original_identity = envelope["identity"]
    coda_revision = it.hub.push_analysis_store(
        pulled,
        ARTIFACT_REPO_ID,
        private=ARTIFACT_PRIVATE,
        token=token,
        provenance={"generated_by": "intervention_from_concept", "iteration_note": "round-trip coda re-upload"},
    )
    refreshed = it.hub.pull_analysis_store(ARTIFACT_REPO_ID, token=token)  # refresh the cached envelope
    coda_envelope = it.hub.describe_analysis_store(ARTIFACT_REPO_ID)
    assert coda_envelope["identity"] == original_identity, "identity must survive re-upload"
    print(f"re-uploaded @ {coda_revision}; identity preserved: {coda_envelope['identity']['store_id']}")
    print(f"iteration note: {coda_envelope['provenance'].get('iteration_note')}")
else:
    print("coda skipped (RUN_REUPLOAD_CODA=False)")

Setting num_proc from 1 back to 1 for the validation split to disable multiprocessing as it only contains one shard.


re-uploaded @ 578e28ae2e7ba1041eed6765c80572c1ad34f087; identity preserved: aadcd8d6-3892-475e-9f8f-1cb21cde2305
iteration note: round-trip coda re-upload


## Summary

- The `it.intervention_from_concept` results became a portable artifact: parquet interchange +
  `it_artifact.json` envelope + generated dataset card, published with `it.hub.push_analysis_store`.
- After deleting all local analysis state, `it.hub.pull_analysis_store` reproduced the exact stored
  columns, hydrated a steering-capable attribution `Graph` through the named backend seam, and
  `it.intervention_from_features` replicated the steering outcome:  the attribution-graph
  generation never re-ran; only the steering forward pass did, from the stored graph.
- Streaming the same store read only the projected columns, through range requests, and decoded them as the full
  pull did.
- Re-uploading kept the artifact's identity while its content fingerprint tracked the change.
